In [0]:
# Config from ../04_utils/shared_config (inlined because %run path resolution fails on serverless)
CATALOG = "workspace"
SCHEMA = "crypto_live"

from pyspark.sql.functions import lit

# Dim_Coin — fairly static reference data (you can enrich this manually later)
coin_names = {
    "bitcoin": "Bitcoin", "ethereum": "Ethereum", "tether": "Tether",
    "binancecoin": "BNB", "solana": "Solana", "ripple": "XRP",
    "cardano": "Cardano", "dogecoin": "Dogecoin", "polkadot": "Polkadot",
    "litecoin": "Litecoin"
}

dim_coin_data = [(coin_id, name) for coin_id, name in coin_names.items()]
dim_coin = spark.createDataFrame(dim_coin_data, ["coin_id", "coin_name"])
dim_coin.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_coin")

# Fact_Crypto_Prices — the time-series fact table, Gold layer
# This is essentially Silver, but joined with Dim_Coin and ready for direct BI/dashboard use
fact_df = (
    spark.table(f"{CATALOG}.{SCHEMA}.silver_crypto_prices")
    .join(dim_coin, on="coin_id", how="left")
    .select("coin_id", "coin_name", "price_usd", "market_cap_usd",
            "volume_24h_usd", "change_24h_pct", "last_updated_ts", "_snapshot_timestamp")
)

fact_df.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.fact_crypto_prices")

print("Gold layer updated.")
display(spark.sql(f"""
    SELECT coin_name, price_usd, change_24h_pct, _snapshot_timestamp
    FROM {CATALOG}.{SCHEMA}.fact_crypto_prices
    ORDER BY _snapshot_timestamp DESC
    LIMIT 10
"""))

Gold layer updated.


coin_name,price_usd,change_24h_pct,_snapshot_timestamp
Dogecoin,0.086332,-3.392421965441063,2026-09-20T16:18:06.112233+00:00
BNB,759.19,-0.9813460267508477,2026-09-20T16:18:06.112233+00:00
Bitcoin,81134.0,-0.7422942528515162,2026-09-20T16:18:06.112233+00:00
Ethereum,2622.88,-0.7591569457511214,2026-09-20T16:18:06.112233+00:00
XRP,1.4,-2.6171259089723136,2026-09-20T16:18:06.112233+00:00
Solana,108.87,-2.451591934282332,2026-09-20T16:18:06.112233+00:00
Cardano,0.225216,-1.1851359852747925,2026-09-20T16:18:06.112233+00:00
Polkadot,1.12,-0.3090399268652494,2026-09-20T16:18:06.112233+00:00
Tether,0.999643,5.813516019974668E-4,2026-09-20T16:18:06.112233+00:00
Litecoin,57.84,-0.3470918928834404,2026-09-20T16:18:06.112233+00:00
